# SoccerTrack Phase 2 Scientific Workflow
Integrated scratch and pretrained player detection and tracking. Cache-first execution; no training, full inference or held-out access. All results are validation117093.

In [ ]:
from pathlib import Path
import json,os,numpy as np,pandas as pd
from IPython.display import display,Image,Markdown
ROOT=Path.cwd()
RUN=ROOT/os.environ.get('SOCCERTRACK_PHASE2_RUN','outputs/phase2_final/phase2_20261125_01')
data=json.loads((RUN/'accepted_results.json').read_text())
print('Accepted config:',data['config_sha256'])
print('Cache-first execution: no fitting or detector inference')
def picture(name):
    display(Image(filename=str(RUN/'figures'/name)))

## 1 Project objective

We detect and track players in panoramic soccer video. Phase 2 compares rule-based vision, a trained HOG/SVM, a trained small CNN and pretrained tiled YOLOv 8 n, using one shared validation protocol. We estimate image-space player boxes and temporary identities, not physiological fatigue.

## 2 Reference and approved scope

We use the paper’s curated bounding-box subset for image-space MOT. GSR/BAS benchmarks differ in task and protocol; no direct leaderboard claim is made. Primary reference: https://arxiv.org/html/2508.01802v2.

## 3 Dataset organization

A clip and ten-column MOT file are joined by match ID. Native images are 4096 x 1080 at 25 FPS. Internal frames are zero-based and boxes use x,y,width,height. Rows describe the 22 on-pitch players. Ball actions, jersey recognition and pitch-space reconstruction are outside the executed scope.

## 4 Data preparation and cleaning

MOT parsing preserves integer identities, rejects malformed/nonfinite values, removes duplicates/degenerate geometry, clips valid intersections and reports issue counts. Sequential decoding supplies requested frames and verifies coverage. Source annotations and pretrained weights remain read-only.

In [ ]:
eda=json.loads((RUN/'eda/stats.json').read_text()); display(pd.DataFrame(eda['matches']).T[['split','boxes','frames_annotated','ids']])

## 5 Exploratory data analysis

The EDA tables summarize only the five approved training matches and validation 117093. Small distant players motivate native-resolution processing, perspective-aware proposals and tiled inference. Linear empirical quantiles use rank(n-1)q. EDA frame counts below distinguish header/dense-annotation agreement from the newly decoded evaluation union.

In [ ]:
picture('eda/boxes_per_match.png'); picture('eda/box_size_hist.png')

## 6 Match split and leakage prevention

Training matches are 118575,118576,118577,118578,128058; validation is 117093. Protected 128057/132831 and disabled 117092 were not opened. Split checks precede data resolution. Selection on validation is disclosed; no independent test performance is claimed.

## 7 Patch sampling and imbalance

Historical training contains 3432 patches:858 positive and 2574 negative, sampled at 39 match/frame pairs; validation contains 704 patches:176/528 over 8 frames. Crops are RGB64 x 32 with context. Proposal negatives repeat: unique pixels are 2782/580. We preserve the fitted data rather than silently deduplicating history.

In [ ]:
refs=json.loads((RUN/'model_references.json').read_text());display(pd.DataFrame(refs['dataset']).T);picture('patch_examples.png');picture('imbalance.png')

## 8 Background and pitch segmentation

The scratch pipeline compares each frame to a label-free median background, masks the pitch in HSV/Lab processing and cleans foreground differences. Background samples span the clip, including future frames: this is an offline, noncausal method. Cached background reuse avoids unnecessary decoding.

In [ ]:
picture('scratch_stages.png');print('Real frame0: raw foreground and morphology are distinct saved stages; no illustrative masks substituted')

## 9 Morphology and candidate proposals

Thresholded differences, median filtering, opening/closing, vertical dilation and connected components form regions. A size model fitted to training data accounts for player height versus image row; filtering and wide-component splitting reduce background/merged blobs. Selected candidate settings are compared with the starting settings on identical eval 10 frames.

In [ ]:
picture('proposal_comparison.png')

## 10 Rule based baseline

Shape, fill and expected perspective size produce a score without a learned classifier. This baseline tests how far domain-specific image processing can go. It is a detection baseline and must not be mislabeled as the majority patch classifier.

## 11 HOG feature representation

Unsigned orientation votes and spatial cells form 756 custom HOG features per 64 x 32 crop. Block normalization reduces contrast dependence. The feature implementation and trained SVM are unchanged. Installed OpenCV lacks HOGDescriptor, so reference numerical parity remains unverified despite internal reproducibility.

In [ ]:
picture('hog_features.png');print('756 custom features; unavailable OpenCV reference API remains disclosed')

## 12 SVM training and calibration

Balanced LinearSVC uses the squared-hinge objective. Training-only stratified folds calibrate scores with a sigmoid; validation patch AP selects C from 0.01/0.1/1/10, with historical C0.01 retained. A calibrated score ranks proposals; good sampled-patch performance alone does not establish detector recall.

## 13 CNN architecture

Three convolution blocks (16/32/64 channels), pooling, global aggregation, dropout and one binary output contain 23873 trainable parameters. RGB normalization is estimated from training patches only. Learned visual features replace fixed HOG features while using the same proposal source.

In [ ]:
picture('cnn_architecture.png');print('23873 trainable parameters; existing checkpoint, no retraining')

## 14 CNN optimization and historical training

The existing model was trained for 8 epochs with Adam (learning rate 0.001, weight decay 0.0001) and weighted BCEWithLogits to address 1:3 imbalance. Training augmentation includes horizontal flips and brightness/contrast changes. Validation AP selected epoch 7. The displayed curves are historical training evidence; no model was retrained during integration.

In [ ]:
picture('cnn_training.png');history=json.loads((ROOT/'outputs/member1_scientific/runs/expanded_20261120T044321992233Z/training_history.json').read_text());display(pd.DataFrame(history))

## 15 Pretrained YOLO comparison

YOLOv 8 n is a pretrained COCO-person reference, not a locally trained Phase 2 network. Its multiscale backbone/head predicts boxes and scores. The source model uses classification and box/distribution losses during pretraining; this project performs inference only and does not invent local YOLO training curves.

## 16 Native tiling and global NMS

1280 x 1080 tiles with 128 px overlap and inference size 1280 retain tiny-player detail. Tile-local boxes are translated/clipped to native coordinates and global NMS at IoU0.5 removes duplicates. Pitch filtering remains off in the accepted selected configuration. Raw cache identity/coverage was verified before shared serialization.

In [ ]:
arch=json.loads((ROOT/'outputs/member2_scientific/runs/member2_validation_20261125/architecture.json').read_text());print('Actual pretrained architecture:',{k:arch.get(k) for k in ['parameters','strides','head','nc']});picture('yolo_stages/tiles.jpg');picture('yolo_stages/tile_local.jpg');picture('yolo_stages/post_nms.jpg');print('Historical full-precision intermediate visuals at the selected configuration; official rounded outputs are evaluated separately')

## 17 IoU Hungarian tracking

IoU measures intersection area divided by union. A gated Hungarian assignment links current boxes to tracks; confirmation and ageing handle transient misses. Every contiguous frame receives an update, including no detections. Position-only association can fragment or swap identities under occlusion.

## 18 ByteTrack association

ByteTrack uses high-confidence detections and lower-score candidates to retain tracks through uncertain observations. Detections down to the 0.05 source floor remain serialized. Reported 0.50 detector threshold is separate from association thresholds. Track IDs are temporary labels; unique-ID counts do not equal identity switches.

## 19 Shared evaluation protocol

E1 classifies sampled patches; E2 classifies every scored proposal and reports many-to-one GT coverage; E3 matches native-frame detections one-to-one in confidence order; E4 evaluates continuous tracks with TrackEval. Tables are read back after 2-decimal boxes/4-decimal scores, so all final metrics use the same representation.

## 20 E1 patch classification

We compare majority negative, HOG/SVM and CNN on the same 704 validation patches. Accuracy, precision, recall, specificity, F1, ROC-AUC and classification AP answer different questions. Majority prediction has 75%accuracy but zero positive recall, illustrating why accuracy alone is misleading.

In [ ]:
table=pd.read_csv(RUN/'tables/E1_accepted.csv');display(table)
assert len(table)>0
print('Loaded accepted E1 table; frame protocol recorded in each Metrics JSON')
picture('eval/roc_classifiers_val.png')
from soccertrack.metrics import binary_classification_metrics
scores=pd.read_csv(ROOT/'outputs/scores/scratch_hog_svm/val/patches.csv')
check=binary_classification_metrics(scores.y_true.to_numpy(),scores.score.to_numpy(),.5)
assert abs(check['metrics']['f1']-data['results'][0]['metrics']['f1'])<.0001
print('Independent saved-score recomputation agrees')

## 21 E2 proposal evaluation

All proposals, including those removed by downstream NMS, are retained for 598 eval 10 frames. Coverage at IoU0.5/0.3 measures whether any candidate can localize each GT box. Classification on natural proposal imbalance is separate. Proposal recall bounds downstream recall under that candidate/localization protocol.

In [ ]:
table=pd.read_csv(RUN/'tables/E2_accepted.csv');display(table)
assert len(table)>0
print('Loaded accepted E2 table; frame protocol recorded in each Metrics JSON')
display(pd.DataFrame([dict(method=d['method'],**d['breakdown']['proposal_recall_at_iou']) for d in data['results'] if d['eval_id']=='E2']))

## 22 E3 common frame detection

Rules, HOG/SVM, CNN and YOLO use the same 598 frame IDs and GT. Precision/recall/F1 use score 0.50 and IoU0.5, with IoU0.3 diagnostics. AP50 and AP50:95 use all detections retained at floor 0.05 and confidence ordering. Our AP is the approved all-point implementation, not an unqualified claim of identical COCO tooling.

In [ ]:
table=pd.read_csv(RUN/'tables/E3_accepted.csv');display(table)
assert len(table)>0
print('Loaded accepted E3 table; frame protocol recorded in each Metrics JSON')
picture('detection_comparison.png');picture('eval/pr_detection_val.png')

## 23 E4 controlled and real tracking

Oracle and seeded noisy GT isolate association behavior for both trackers. Noise drops 10%of boxes and perturbs coordinates by Gaussian 2 px with seed 42; repaired boxes stay valid. Real selected scratch+IoU and YOLO+ByteTrack use 1500 contiguous frames. TrackEval supplies HOTA/DetA/AssA/LocA/IDF1/MOTA/IDSW. Controlled ByteTrack rows use starting settings; the real YOLO row selects conservative settings (high/new 0.4,low 0.1,buffer 30,match 0.8) from the three preplanned candidates by HOTA, then IDF1/MOTA. Explicit tracker overrides are stored separately from the base shared configuration; no detector reinference or new grid was introduced.

In [ ]:
table=pd.read_csv(RUN/'tables/E4_accepted.csv');display(table)
assert len(table)>0
print('Loaded accepted E4 table; frame protocol recorded in each Metrics JSON')
picture('tracking_comparison.png')

## 24 Model comparison and selection

The selected scratch detector is scratch_cnn, using highest E3 F1 at the fixed 0.50 reporting threshold on validation. This is validation selection, not an independent generalization result. We retain the existing models and selected proposal configuration instead of introducing a new optimization search.

## 25 Error analysis

Synchronized native-frame views expose distant players, background components, merged players and localization errors. Error-example CSVs use the shared one-to-one matcher. Red boxes mark missed GT; crops magnify actual validation pixels. TrackEval summary errors are global sequence measures; a single still image does not establish an identity switch.

In [ ]:
picture('synchronized_frame_0.png');picture('representative_detection_error.png');picture('tracking_timeline.png');picture('tracking_transition.png');print('Tracking transition views are qualitative matched-ID diagnostics, not official CLEAR IDSW event labels');display(pd.read_csv(RUN/'tables/error_examples.csv').groupby('method')[['tp','fp','fn']].sum())

## 26 Runtime and resources

Scratch methods share decoding and proposals. Per-method processing seconds include measured shared preparation plus scoring/NMS; published frameset FPS allocates union processing proportionally and excludes decode/background/setup/write. Union wall time is measured separately. Historical YOLO warm processing 8.1282 FPS excludes setup/decode/write. These are different scopes; no realtime or fair end-to-end speed ranking is claimed.

In [ ]:
coverage=json.loads((RUN/'coverage.json').read_text());display(pd.DataFrame([{'union_frames':coverage['decoded_union_frames'],'union_wall_seconds':coverage['union_wall_seconds']}]))
print('Processing seconds by method:',coverage['processing_seconds_by_method']);print('These scopes differ from historical warm YOLO processing FPS')

## 27 Reference comparison

Paper Table 3 reports GS-HOTA12.25 over 20 halves and 10.96 on test halves. Our 60 second image-space HOTA differs in coordinates, attributes and sequence length. Direct numerical superiority is not claimed.

## 28 Scientific interpretation

YOLO achieves E3 F1 0.8162, versus CNN 0.7174 and HOG/SVM 0.7093 on identical frames. The scratch selection is scratch_cnn; compare precision and recall as well as this narrow validation F1 difference. Proposal coverage changes from 0.5412 to 0.7083 at IoU0.5. Selected YOLO/ByteTrack gives HOTA 0.4321, IDF1 0.4327 and MOTA 0.3255; selected scratch/IoU gives 0.2551, 0.268 and 0.4058, respectively. Different metrics measure different errors; no universal winner or independent generalization claim follows.

## 29 Limitations

One validation match, limited sampled training coverage, repeated negatives, validation-based selection, offline future-frame background, unresolved OpenCV HOG reference verification, small/occluded players, temporary tracking identities and differing runtime scopes limit conclusions. No physiological fatigue labels or held-out results support clinical or generalization claims.

## 30 Phase 3 and presentation

Future work may improve proposals/appearance association, test explainability, perform authorized held-out assessment and investigate calibrated pitch-space workload proxies. None is implemented as Phase 3 work here. The final PowerPoint is intentionally pending, so complete academic submission readiness is not claimed.

## 31 Reproduction and artifact identity

Normal notebook execution only loads accepted saved artifacts. Reproduction uses the production config/registry, existing hash-checked models, verified YOLO migration and checkpointed missing scratch inference. EXECUTION_JOURNAL and stage.json record progress. All historical experiments remain immutable and the user manages Git.

In [ ]:
from soccertrack import config
assert config.config_sha256()==data['config_sha256']
print('Current shared configuration matches accepted run')
print('Explicit expensive reproduction: see START_HERE.md and workflow.py stages; never automatic')

## 32 Conclusion

The integrated project provides reproducible validation comparisons across classical vision, trained handcrafted/learned patch classifiers, pretrained detection and two trackers. Conclusions below are derived from accepted E1–E4 artifacts. Manual review should inspect the application, synchronized video, notebook and report before presentation.